# 🎮 Video Games Sentiment Analysis (Analisis Sentimen Ulasan Video Games)

Notebook ini mengimplementasikan alur lengkap analisis sentimen berbasis **Natural Language Processing (NLP)** dan **Machine Learning** pada dataset **Amazon Reviews 2023** kategori **Video_Games**.

### Alur Proyek:
1. **Persiapan Lingkungan & Pustaka**
2. **Pengunduhan & Pemuatan Data Bertahap (Streaming / Chunking)**
   - Membaca data `.jsonl.gz` baris demi baris menggunakan `json.loads` agar memori RAM tidak penuh.
   - Memfilter ulasan selama pembacaan: hanya ulasan dengan `timestamp <= 1577836799999` (hingga 31 Desember 2019 UTC).
3. **Exploratory Data Analysis (EDA)**
4. **Pra-pemrosesan Teks (Text Preprocessing)**
5. **Ekstraksi Fitur (TF-IDF Vectorization)**
6. **Pelatihan Model Klasifikasi (Logistic Regression Pipeline)**
7. **Evaluasi Model** (Accuracy, Precision, Recall, F1-Score & Confusion Matrix)
8. **Penyimpanan Model Terlatih** (`model/model_sentimen.joblib`)
9. **Pengujian Inferensi Real-time**

## 1. Persiapan Lingkungan & Pemuatan Pustaka

In [1]:
import os
import re
import gzip
import json
import zlib
import time
import datetime
import requests
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_theme(style="whitegrid")
print("Pustaka berhasil dimuat!")

Pustaka berhasil dimuat!


## 2. Pengunduhan & Pemuatan Data secara Bertahap (Chunking/Streaming)

> **Penting:** Dataset Amazon Reviews 2023 kategori `Video_Games` berukuran besar (arsip terkompresi `.jsonl.gz` berukuran ~814 MB dan beberapa gigabyte ketika diekstrak). Untuk mencegah kehabisan memori RAM, kita membaca berkas secara **streaming baris demi baris (line-by-line)** menggunakan `json.loads`.
>
> **Aturan Filter:** Hanya ulasan dengan `timestamp <= 1577836799999` (31 Desember 2019, 23:59:59.999 UTC) yang diproses.

In [2]:
URL_VIDEO_GAMES = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Video_Games.jsonl.gz"
CUTOFF_TIMESTAMP = 1577836799999  # 31 Desember 2019 23:59:59.999 UTC

def stream_and_filter_reviews(url=URL_VIDEO_GAMES, max_samples=5000, cutoff_timestamp=CUTOFF_TIMESTAMP):
    """
    Membaca berkas .jsonl.gz secara streaming lewat HTTP chunks,
    mendekompresi on-the-fly, membaca baris demi baris dengan json.loads,
    serta memfilter data ulasan dengan timestamp <= cutoff_timestamp.
    """
    print(f"Mulai streaming dari: {url}")
    print(f"Filter batas timestamp: <= {cutoff_timestamp} (Hingga 31 Des 2019)")
    
    start_time = time.time()
    resp = requests.get(url, stream=True, timeout=60)
    resp.raise_for_status()
    
    d = zlib.decompressobj(16 + zlib.MAX_WBITS)
    records = []
    buffer = ""
    total_scanned = 0
    
    for chunk in resp.iter_content(chunk_size=65536):
        if not chunk:
            continue
        decompressed = d.decompress(chunk)
        buffer += decompressed.decode("utf-8", errors="ignore")
        lines = buffer.split("\n")
        buffer = lines[-1]
        
        for line in lines[:-1]:
            line_str = line.strip()
            if not line_str:
                continue
            total_scanned += 1
            record = json.loads(line_str)
            ts = record.get("timestamp")
            
            # Filter baris: timestamp <= 31 Des 2019
            if ts is not None and ts <= cutoff_timestamp:
                text = record.get("text", "")
                if text and len(text.strip()) > 5:
                    rating = float(record.get("rating", 0))
                    dt = datetime.datetime.fromtimestamp(ts / 1000.0, datetime.timezone.utc)
                    
                    # Sentimen: 1 (Positif, rating >= 4), 0 (Negatif, rating <= 2)
                    sentiment = 1 if rating >= 4.0 else (0 if rating <= 2.0 else None)
                    
                    records.append({
                        "rating": rating,
                        "title": record.get("title", ""),
                        "text": text.strip(),
                        "timestamp": ts,
                        "date": dt.strftime("%Y-%m-%d %H:%M:%S"),
                        "year": dt.year,
                        "asin": record.get("asin", record.get("parent_asin", "")),
                        "verified_purchase": record.get("verified_purchase", False),
                        "helpful_vote": record.get("helpful_vote", 0),
                        "sentiment": sentiment
                    })
            
            if len(records) >= max_samples:
                break
        if len(records) >= max_samples:
            break
            
    elapsed = time.time() - start_time
    print(f"Selesai! Memindai {total_scanned} baris, mendapatkan {len(records)} ulasan tersaring dalam {elapsed:.2f} detik.")
    return pd.DataFrame(records)

In [3]:
# Pemuatan ulasan tersaring (menggunakan file lokal tersimpan atau streaming)
filtered_csv_path = os.path.join("data", "video_games_reviews_2019.csv")

if os.path.exists(filtered_csv_path):
    print(f"Memuat data ulasan tersaring dari berkas lokal: {filtered_csv_path}")
    df = pd.read_csv(filtered_csv_path)
else:
    df = stream_and_filter_reviews(max_samples=5000)
    os.makedirs("data", exist_ok=True)
    df.to_csv(filtered_csv_path, index=False)

# Verifikasi aturan timestamp <= 31 Desember 2019 (1577836799999)
print(f"Total baris data : {len(df)}")
print(f"Maksimum timestamp : {df['timestamp'].max()} (Batas: {CUTOFF_TIMESTAMP})")
print(f"Rentang tanggal    : {df['date'].min()} s/d {df['date'].max()}")
print("Semua data <= 2019 :", (df['timestamp'] <= CUTOFF_TIMESTAMP).all())
df.head()

## 3. Exploratory Data Analysis (EDA)
Menganalisis distribusi rating, sentimen biner (Positif = 1, Negatif = 0), dan statistik panjang kata ulasan.

In [4]:
# Filter ulasan dengan sentimen biner (abaikan ulasan netral rating 3)
df_sentiment = df.dropna(subset=["sentiment"]).copy()
df_sentiment["sentiment"] = df_sentiment["sentiment"].astype(int)
df_sentiment["word_count"] = df_sentiment["text"].apply(lambda x: len(str(x).split()))

print("Distribusi Rating Asli (1 - 5 bintang):")
print(df["rating"].value_counts().sort_index())

print("\nDistribusi Sentimen Biner:")
print(df_sentiment["sentiment"].value_counts())

# Visualisasi EDA
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Distribusi Rating
sns.countplot(data=df, x="rating", ax=axes[0], palette="viridis")
axes[0].set_title("Distribusi Rating Ulasan Video Games", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Jumlah Ulasan")

# 2. Distribusi Sentimen
sns.countplot(data=df_sentiment, x="sentiment", ax=axes[1], palette=["#e74c3c", "#2ecc71"])
axes[1].set_xticklabels(["Negatif (0)", "Positif (1)"])
axes[1].set_title("Distribusi Kelas Sentimen", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Jumlah Ulasan")

# 3. Distribusi Tren Ulasan per Tahun
year_counts = df["year"].value_counts().sort_index()
sns.lineplot(x=year_counts.index, y=year_counts.values, marker="o", ax=axes[2], color="#3498db")
axes[2].set_title("Tren Ulasan per Tahun (Hingga 2019)", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Tahun")
axes[2].set_ylabel("Jumlah Ulasan")

plt.tight_layout()
plt.show()

## 4. Pra-pemrosesan Teks (Preprocessing)
Menghapus URL, tanda baca, simbol aneh, dan melakukan case folding ke huruf kecil (lowercase).

In [5]:
def clean_review_text(text):
    """Membersihkan teks ulasan game."""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df_sentiment["clean_text"] = df_sentiment["text"].apply(clean_review_text)
df_sentiment[["text", "clean_text", "sentiment"]].head()

## 5. Pembagian Data Latih dan Data Uji (Train & Test Split)
Membagi dataset ulasan menjadi 80% data latih dan 20% data uji secara terstratifikasi.

In [6]:
X = df_sentiment["clean_text"]
y = df_sentiment["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Data Latih (Train): {len(X_train)} sampel")
print(f"Data Uji (Test)   : {len(X_test)} sampel")

## 6. Pipeline Pemodelan (TF-IDF + Logistic Regression)
Membangun pipeline scikit-learn terintegrasi: `TfidfVectorizer` (unigram & bigram) dan pengklasifikasi `LogisticRegression`.

In [7]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),
        stop_words="english",
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

# Pelatihan model
pipeline.fit(X_train, y_train)
print("Pelatihan Model Pipeline selesai!")

## 7. Evaluasi Model
Menghitung metrik Akurasi, Precision, Recall, F1-Score, dan visualisasi Confusion Matrix Heatmap.

In [8]:
y_pred = pipeline.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Akurasi Model: {acc * 100:.2f}%")
print("\nLaporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=["Negatif (0)", "Positif (1)"]))

# Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred Negatif", "Pred Positif"],
            yticklabels=["Aktual Negatif", "Aktual Positif"])
plt.title("Confusion Matrix Klasifikasi Sentimen Game", fontsize=12, fontweight="bold")
plt.ylabel("Kelas Aktual")
plt.xlabel("Kelas Prediksi")
plt.tight_layout()
plt.show()

## 8. Penyimpanan Model (Model Serialization)
Menyimpan model pipeline ke berkas `model/model_sentimen.joblib`.

In [9]:
os.makedirs("model", exist_ok=True)
model_path = os.path.join("model", "model_sentimen.joblib")
joblib.dump(pipeline, model_path)
print(f"Model berhasil disimpan di: {model_path}")

## 9. Pengujian Inferensi Real-time (Uji Ulasan Baru)
Menguji model dengan berbagai contoh ulasan game nyata dari pemain.

In [10]:
loaded_model = joblib.load(model_path)

ulasan_uji = [
    "Incredible graphics, captivating story, and the combat gameplay is buttery smooth!",
    "Constant crashing on launch, terrible optimization, completely unplayable waste of money.",
    "Super fun multiplayer experience with friends, highly recommended for RPG fans.",
    "Overpriced buggy mess loaded with greedy microtransactions and pay to win schemes."
]

prediksi = loaded_model.predict(ulasan_uji)
probabilitas = loaded_model.predict_proba(ulasan_uji)

for teks, pred, prob in zip(ulasan_uji, prediksi, probabilitas):
    label = "POSITIF" if pred == 1 else "NEGATIF"
    conf = max(prob) * 100
    print(f"Ulasan    : \"{teks}\"")
    print(f"Sentimen  : {label} (Keyakinan: {conf:.2f}%)")
    print("-" * 70)